<!-- <a href="https://colab.research.google.com/github/prane-eth/iRAT/blob/main/Result-filter/3 - Result-filter - pretrained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> -->

## Result filter module (_Attention-Retrieval_)

<!-- - Mostly, each query has only upto 2 selections.
	- Sometimes, a score of 5.0 is not selected because of better options.
	- So, don't train a classifier using scores.
	- Create own method. 
	- Get top 3 scores, and ensure they have a good score (3.0+ for miniLM, <0.2 for some other model).
	- As the case is complex, use a large neutral network. Can be made smaller later.
	- Neural network can analyse all the scores to decide to select some of them. -->

![Result-filter workflow](3-Result-filter_workflow.png)

## Initialization

### Setup directories

In [1]:
import sys
import os
if 'google.colab' in sys.modules:
	# !pip install --upgrade datasets sentence_transformers
	!pip install sentence_transformers
	from IPython.display import clear_output
	clear_output()
	os.chdir('iRAT')

data_dir = 'data'
if os.path.basename(os.getcwd()) != data_dir:
	os.chdir('notebooks')
	os.chdir('Result-filter')
	os.chdir('4-Attention-Retrieval')
	os.chdir(data_dir)

### Load the dataset

In [2]:
import numpy as np
import random
import torch

# set random seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
	torch.cuda.manual_seed_all(seed)
	# for fully deterministic (seeded) CuDNN behavior (slower), you can also do:
	# torch.backends.cudnn.deterministic = True
	# torch.backends.cudnn.benchmark = False

if not os.path.exists('coding_dataset'):
	# os.system('python 2_create_dataset.py')
	raise ValueError('Dataset directory "coding_dataset" does not exist. Please run 2_create_dataset.py to create the dataset.')


from datasets import load_from_disk

# restores the same DatasetDict with train/validation splits
dataset = load_from_disk('coding_dataset')
if not len(dataset['train']) or not len(dataset['validation']):
	raise ValueError('The training dataset is empty. Please check the dataset creation process.')

print('Sample rows:')
for val_index, row in enumerate(dataset['train']):
	if val_index > 1:  # Display only the first 5 rows
		break
	print(f'Row {val_index}')
	print(f'  Query: {row["query"]}')

Sample rows:
Row 0
  Query: what is an of clause sql
Row 1
  Query: javascript define array


### Load the model

In [3]:
from sentence_transformers import CrossEncoder

model_name = 'cross-encoder/ms-marco-MiniLM-L6-v2'  # 22.7M params - 3.3M downloads on Huggingface
model_name_short = 'MiniLM-L6-v2'

# model_name = 'cross-encoder/ms-marco-MiniLM-L12-v2'  # 33.4M params
# model_name_short = 'MiniLM-L12-v2'

# model_name = 'mixedbread-ai/mxbai-rerank-xsmall-v1'  # 70.8M params - 1.8M downloads
# model_name_short = 'mxbai-rerank-xsmall-v1'

model = CrossEncoder(model_name)

def predict(passages: list[str], query: str, top_k: int) -> list[float]:
	if 'mxbai' in model_name:
		results = model.rank(query, passages, return_documents=True, top_k=top_k)
		if len(results) != top_k:
			raise ValueError(f'Expected {top_k} results, got {len(results)}')
		# [{'corpus_id': 2, 'score': 0.94370663, 'text': '....'}, ....]
		scores = [0] * len(passages)
		for result in results:
			scores[result['corpus_id']] = result['score']
		return scores

	# Query the LLM with a list of passages and a query.
	# Returns a list of scores for each passage.
	scores = model.predict([(query, passage) for passage in passages], batch_size=1)
	scores = scores.tolist()
	return scores

### Define functions related to the scores

In [ ]:
results = {}

def mark_as_correct(index, score=True):
	results[str(index)] = score
	# save_results(results)
	# print(f'Correct')

def mark_as_incorrect(index, comment=None):
	results[str(index)] = False
	# save_results(results)
	# print(f'Incorrect')
	# if comment:
	# 	print('\t -', comment)

## Create training data for classification
to predict the relevance of the results based on scores generated by the pretrained model.

In [5]:
# Generate training data to accept the ranks as input and return True/False as output.
import json
import os
data_filename = f'classifier_dataset-{model_name_short}.json'

if os.path.exists(data_filename):
	print(f'Loading existing data...')
	with open(data_filename, 'r') as f:
		data = json.load(f)
	all_selection_statuses = data['selection_statuses']
	all_scores = data['scores']
else:
	print(f'Generating new data...')
	all_selection_statuses = {}
	all_scores = {}
	for subset in dataset:
		if subset == 'test':
			print(f'Skipping subset "{subset}" as it is not used for any process.')
			continue
		all_selection_statuses[subset] = []
		all_scores[subset] = []
		for val_index, row in enumerate(dataset[subset]):
			if val_index % 100 == 0:
				print(f'Processing {subset} row {val_index} of {len(dataset[subset])}')
			selection_statuses = row['passages']['is_selected']
			all_selection_statuses[subset].append(selection_statuses)
			predictions = predict(row['passages']['passage_text'], row['query'])
			all_scores[subset].append(predictions)
		if len(all_selection_statuses[subset]) != len(all_scores[subset]):
			raise ValueError(f'The number of selection statuses does not match the number of scores in subset {subset}.')

	# Save the data to the file
	with open(data_filename, 'w') as f:
		data = {
			'selection_statuses': all_selection_statuses,
			'scores': all_scores,
		}
		json.dump(data, f, indent=4)
	print(f'Data saved to {data_filename}')

print(all_selection_statuses['train'][0][:5])  # [[0, 1, ....], ...]
print(all_scores['train'][0][:5])  # [[0.123, 0.456, ...], ...]

# count the lengths of inputs
max_passages = 0
for row in dataset['train']:
	count = len(row['passages']['is_selected'])
	max_passages = max(max_passages, count)
print(f'Maximum number of passages in a row: {max_passages}')  # 10

Loading existing data...
[0, 0, 0, 0, 0]
[3.7543892860412598, -5.701083183288574, 3.0622544288635254, -6.228157043457031, 3.720428466796875]
Maximum number of passages in a row: 10


## Create a classifier
to predict the selection of paragraphs based on the scores

In [6]:
if 'MiniLM' in model_name:
	score_threshold = 8.0
elif 'mxbai' in model_name:
	score_threshold = 0.8  # rank 1 is the best, so we want to select passages with rank < 1.0
else:
	raise ValueError(f'Thresholds not defined for model {model_name_short}.')

# top_n = 3

def classify(predictions: list[float], correct_count: int) -> list[bool]:
	if not predictions:
		raise ValueError('Model predictions list is empty')
	top_n = correct_count
	scores_dict = { index: score for index, score in enumerate(predictions)
					if score > score_threshold }
	top_scores = sorted(scores_dict.items(), key=lambda x: x[1], reverse=True)[:top_n]
	selected_indices = [index for index, score in top_scores]
	# return True for selected indices, False for others
	result = [index in selected_indices for index in range(len(predictions))]
	if len(result) != len(predictions):
		raise ValueError(f'Length of result {len(result)} does not match length of scores {len(predictions)}')
	return result

## Generate predictions

In [7]:
subset = 'validation'  # 'test' subset in MARCO doesn't mention the correct answers.

def evaluate_selections(selected_indices, correct_indices, val_index):
	if not correct_indices:
		# no need to select anything, so consider it correct
		mark_as_correct(val_index)
		return
	total_correct = len(correct_indices)
	correct_selected = len([index for index in selected_indices if index in correct_indices])
	score = correct_selected / total_correct
	if score:
		mark_as_correct(val_index, score)
	else:
		mark_as_incorrect(val_index, 'None of the correct indices were selected')

for val_index, row in enumerate(dataset[subset]):
	# print('---', val_index)

	# Get the correct indices from the dataset
	correct_indices = [index for index, x in enumerate(row['passages']['is_selected']) if x > 0]
	# print(f'Correct indices: {correct_indices}')
	if not correct_indices:
		# No correct indices. We need not select anything.
		evaluate_selections([], correct_indices, val_index)
		continue

	# Select indices using the model
	correct_count = len(correct_indices)
	if 'mxbai' in model_name:
		predictions = predict(row['passages']['passage_text'], row['query'], top_k=correct_count)
	else:
		predictions = all_scores[subset][val_index]  # Load the cached predictions
	# print('Predictions:', predictions)
	classifications = classify(predictions, correct_count)
	selected_indices = []
	for index, pred in enumerate(classifications):
		if pred:
			# print(index, '-', pred, 'Prediction:', predictions[index])
			selected_indices.append(index)
	# print(f'Selected indices: {selected_indices}')

	evaluate_selections(selected_indices, correct_indices, val_index)

if True:
	total_evaluated = len(results)
	total_correct = sum(results.values())
	print(f'Total evaluated: {total_evaluated}, Total correct: {total_correct}')
	accuracy = total_correct / total_evaluated if total_evaluated > 0 else 0
	print(f'Accuracy: {accuracy:.2%}')
	print(f'Model: {model_name}')

	with open(os.path.join('..', 'scores.txt'), 'a') as f:
		f.write(f'{model_name_short} - accuracy: {accuracy:.2%}\n')

Total evaluated: 270, Total correct: 197.0
Accuracy: 72.96%
Model: cross-encoder/ms-marco-MiniLM-L6-v2
